# Проблем

## Предистория
Водеща компания за социални медии е разработила сложна система за класификация на съдържанието, която помага за организиране и категоризиране на генерираното от потребителите съдържание в тяхната платформа. Най-новата им технология за изкуствен интелект произвежда висококачествени вграждания за текстово съдържание (което ще симулираме с помощта на вграждания Sentence-BERT в тази задача).

## Предизвикателството
Въпреки че настоящата система работи отлично в сървърна инфраструктура, компанията има за цел да пренесе част от тези възможности за класификация директно на мобилните устройства на потребителите. Това изисква значително намаляване на размерите на вграждането, като същевременно се запазва възможно най-голяма част от първоначалната способност за клъстериране.

## Вашата задача
Вашата задача е да разработите функция за преобразуване, която може да преобразува оригиналните 768-измерни вграждания в 32-измерни вграждания, като същевременно запазва основната информация, необходима за точното клъстеризиране на съдържанието.

## Правила
- Не променяйте предоставените клетки, различни от отбелязаните за вашата реализация.
- Обучението на модели е разрешено, но директното използване на K-Means или KNN във функцията за преобразуване е *забранено*.
- Можете да оценявате с по-малко или повече от 10 изпълнения, но окончателните заявки ще бъдат оценени с 10 изпълнения.
- Подсказка: Опитайте се да направите решението си възможно най-стабилно на случайни инициализации.



# Setup

In [ ]:
import os
import torch
import pickle
import random
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from sklearn.datasets import fetch_20newsgroups
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

In [ ]:
def set_seed(seed_value = 42):
  random.seed(seed_value)
  np.random.seed(seed_value)
  torch.manual_seed(seed_value)
  if torch.cuda.is_available():
      torch.cuda.manual_seed(seed_value)
      torch.cuda.manual_seed_all(seed_value)
      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

set_seed()

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, model_name='all-MiniLM-L6-v2'):
        self.texts = texts
        self.labels = labels
        self.model = SentenceTransformer(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        with torch.no_grad():
            emb = self.model.encode(text, convert_to_tensor=True)
        label = self.labels[idx]
        return emb, label

In [ ]:
def load_and_process_split(split_name, model_name='all-MiniLM-L6-v2'):
    with open(f"{split_name}_texts.pkl", "rb") as f:
        texts, labels = pickle.load(f)

    dataset = TextDataset(texts, labels, model_name=model_name)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    X = []
    y = []
    for emb, label in tqdm(loader):
        X.append(emb.cpu().numpy())
        y.append(label.numpy())

    return np.vstack(X), np.concatenate(y)

# Main code

In [ ]:
X768, y = load_and_process_split("public")
k = 4

100%|██████████| 125/125 [00:25<00:00,  4.86it/s]


In [ ]:
set_seed()
km768 = KMeans(n_clusters=k, random_state=0).fit(X768)
public_clusters = km768.labels_

In [ ]:
def transform(X_high):
    assert X_high.ndim == 2 and X_high.shape[1] >= 32, "Input must be 2D with >=32 dims."
    ### TODO: YOUR CODE HERE
    out = X_high[:, :32]
    ###

    assert out.ndim == 2 and out.shape[1] == 32, f"Embeddings must be shape (N,32), got {out.shape}."
    return out

In [ ]:
def evaluate_clustering(ref_labels, n_clusters, n_runs=10):
    nmi_scores = []
    for seed in range(n_runs):
        set_seed(seed)
        X32 = transform(X768)
        km = KMeans(n_clusters=n_clusters, random_state=0).fit(X32)
        pred = km.labels_
        nmi = normalized_mutual_info_score(ref_labels, pred)
        nmi_scores.append(nmi)

    mean_nmi = np.mean(nmi_scores)
    std_nmi = np.std(nmi_scores)
    return mean_nmi, std_nmi

mean_nmi, std_nmi = evaluate_clustering(public_clusters, k)
print(f"\nPublic NMI (first-32 dims): {mean_nmi:.4f} ± {std_nmi:.4f}")


Public NMI (first-32 dims): 0.2487 ± 0.0000


## Understanding the Evaluation Metric: NMI

Normalized Mutual Information (NMI) is a measure that tells us how well two different clustering assignments agree with each other. It's particularly well-suited for this challenge for several reasons:

- **Scale Independence**: NMI is normalized between 0 (no mutual information) and 1 (perfect correlation), making it easy to interpret regardless of the number of clusters or data points.

- **Permutation Invariance**: NMI doesn't require the cluster labels to match exactly - it only cares about the overall grouping structure. This is important because k-means can assign different numerical labels to the same logical clusters in different runs.

In our case, we use NMI to compare:
- The clustering obtained from the original 768-dimensional embeddings (reference)
- The clustering obtained from your transformed 32-dimensional embeddings

A higher NMI score means your transformation better preserves the original clustering structure, which is exactly what we want for the mobile deployment scenario.
